# 🧪 PT-W2-D5 概念实验：Capability 抽取——BCM 到 LangChat 映射

> 配套阅读：`PT-W2-D5-Capability抽取-BCM到LangChat映射.md`
> 从 BCM 能力行抽取原子能力，映射到 LangChat SkillRelease。

## 第 1 格：Ontology Capability 声明

In [ ]:
from dataclasses import dataclass, field

@dataclass
class OntologyCapability:
    name: str
    domain: str
    description: str
    actor: str
    input_fields: list[str]
    precondition: list[str]
    effect: list[str]
    governed_by: str

caps = [
    OntologyCapability("space.split", "资源管理", "拆分铺位", "招商",
                        ["source_space_id", "target_spaces"],
                        ["source.status == vacant", "no_active_lease(source)"],
                        ["create target_spaces"],
                        "workflow-approvals"),
    OntologyCapability("lease.create", "合同管理", "创建租赁合同", "招商",
                        ["merchant_id", "resource_id", "terms"],
                        ["resource.status == available"],
                        ["occupancy-effect", "financial-effect"],
                        "workflow-approvals"),
    OntologyCapability("lease.terminate", "合同管理", "终止合同", "运营",
                        ["contract_id", "termination_type"],
                        ["inspection completed"],
                        ["occupancy-effect", "financial-effect"],
                        "workflow-approvals"),
    OntologyCapability("billing.generate", "财务管理", "生成账单", "财务",
                        ["contract_id", "period"],
                        ["contract.status == active"],
                        ["financial-effect"],
                        "none"),
    OntologyCapability("approval.submit", "工作流", "提交审批", "any",
                        ["business_type", "payload"],
                        [],
                        ["state-transition-effect"],
                        "K2/BPM"),
]

for c in caps:
    print(f"  {c.name:<18} domain={c.domain}")
    print(f"    precondition: {' ∧ '.join(c.precondition) or '无'}")
    print()

## 第 2 格：BCM → LangChat 映射链

In [ ]:
@dataclass
class SkillRelease:
    name: str
    capability_ref: str
    trigger_event: str
    provider: str

skill_releases = [
    SkillRelease("创建租赁合同", "lease.create", "招商签约", "MI.lease-management"),
    SkillRelease("合同终止清算", "lease.terminate", "终止申请审批通过", "MI.lease-management"),
    SkillRelease("出账操作", "billing.generate", "账期触发", "MI.billing"),
    SkillRelease("铺位拆合操作", "space.split", "招商需求", "MI.space-management"),
    SkillRelease("审批提交", "approval.submit", "业务操作触发", "K2/BPM"),
]

print("BCM Capability → Ontology Capability → SkillRelease 映射：")
print(f"{'SkillRelease':<14} {'Ontology Cap':<18} {'Provider':<20}")
print("-" * 55)
for s in skill_releases:
    print(f"{s.name:<14} {s.capability_ref:<18} {s.provider:<20}")

## 第 3 格：Agent 行动空间检查

In [ ]:
# Agent 检查：当前状态下可调用哪些 Capability
def available_capabilities(caps, state_facts):
    available = []
    for c in caps:
        ok = True
        for pre in c.precondition:
            # 简化求值
            if "status == " in pre:
                _, val = pre.split("status == ")
                key = c.input_fields[0].split("_")[0]
                if state_facts.get(key) != val.strip():
                    ok = False
            elif "no_active" in pre:
                if state_facts.get("active_lease", True):
                    ok = False
        if ok:
            available.append(c.name)
    return available

# 场景：A101 空置、无活跃合同
state = {"resource": "vacant", "active_lease": False}
avail = available_capabilities(caps, state)
print(f"A101 空置时可用能力: {avail}")

# 场景：A101 使用中、有活跃合同
state2 = {"resource": "in-use", "active_lease": True}
avail2 = available_capabilities(caps, state2)
print(f"A101 使用中可用能力: {avail2}")
print("\nAgent 不能直接创建合同到使用中的铺位（precondition 不满足）")

## 第 4 格：可视化——领域 × 能力热力图

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")
font_name = font_manager.FontProperties(fname="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc").get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

domains = ["资源管理", "合同管理", "财务管理", "工作流", "招商管理", "运营管理"]
cap_groups = ["create", "modify", "terminate", "query", "approval"]
# 模拟热力图数据
heatmap = np.array([
    [2,1,0,1,1],  # 资源
    [2,2,1,1,1],  # 合同
    [2,1,1,1,1],  # 财务
    [0,0,0,0,3],  # 工作流
    [2,1,0,1,1],  # 招商
    [1,1,1,1,0],  # 运营
])

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(heatmap, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(cap_groups)))
ax.set_xticklabels(cap_groups)
ax.set_yticks(range(len(domains)))
ax.set_yticklabels(domains)
for i in range(len(domains)):
    for j in range(len(cap_groups)):
        ax.text(j, i, str(heatmap[i,j]), ha="center", va="center")
plt.colorbar(im, ax=ax, label="原子能力数量")
ax.set_title("MI CRE 领域 × 能力热力图（示例）")
plt.tight_layout()
plt.savefig("/root/learning-notebooks/第10周/d5_capability.png", dpi=100)
plt.show()
print("能力热力图已绘制")